# RetinaSafe — Step 4: RETFound smoke test

**Goal:** confirm we can load the RETFound CFP weights and extract features from BRSET images. No training, no architecture work yet — just "can we get the model running."

**Why this is its own step:** RETFound was trained with MAE (masked autoencoder), and the released `.pth` may have key naming that doesn't directly match timm's standard ViT keys. Loading it cleanly is its own debugging task. Better to isolate it now than discover it during a 2-hour training run.

**Inputs to attach (right sidebar → + Add Input):**
1. `samarthmishra208/retfound-mae-nature-cfp-weights` (the weights you just uploaded)
2. `samarthmishra208/brset-brazilian-multilabel-ophthalmological` (for the 5 sample images)

**Settings:** GPU T4 (small workload — inference on 5 images), Internet **off**.

**Expected runtime:** ~1 min.

## Cell 1 — Imports + locate inputs

In [ ]:
import os, glob, json, pathlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:", torch.__version__, "GPU:", torch.cuda.is_available(),
      "Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# Locate RETFound weights
pth_candidates = glob.glob("/kaggle/input/**/RETFound_mae_natureCFP.pth", recursive=True)
if not pth_candidates:
    pth_candidates = glob.glob("/kaggle/input/**/*.pth", recursive=True)
print("\nFound .pth files:")
for p in pth_candidates: print(" ", p)
assert pth_candidates, "RETFound .pth not found"
WEIGHTS = pth_candidates[0]

# Locate BRSET (for sample images)
brset_candidates = glob.glob("/kaggle/input/**/fundus_photos", recursive=True)
print("\nFound fundus_photos at:")
for p in brset_candidates: print(" ", p)
assert brset_candidates, "BRSET fundus_photos not found"
IMAGES_DIR = brset_candidates[0]

print(f"\nWEIGHTS: {WEIGHTS}")
print(f"IMAGES_DIR: {IMAGES_DIR}")

## Cell 2 — Inspect the .pth structure

MAE-pretrained weights are often nested in a dict like `{"model": {...}}` or `{"state_dict": {...}}`. Find the actual weights tensor dict before trying to load.

In [ ]:
raw = torch.load(WEIGHTS, map_location="cpu", weights_only=False)
print(f"Top-level type: {type(raw)}")
if isinstance(raw, dict):
    print(f"Top-level keys: {list(raw.keys())[:20]}")
    # Try to find the actual state_dict
    if "model" in raw and isinstance(raw["model"], dict):
        state_dict = raw["model"]
        print("\n-> Using raw['model']")
    elif "state_dict" in raw and isinstance(raw["state_dict"], dict):
        state_dict = raw["state_dict"]
        print("\n-> Using raw['state_dict']")
    elif all(isinstance(v, torch.Tensor) for v in list(raw.values())[:5]):
        state_dict = raw
        print("\n-> Top-level dict IS the state_dict")
    else:
        # Show what's in there to help diagnose
        print("\n!! Could not auto-detect state_dict. Top-level structure:")
        for k, v in raw.items():
            print(f"  {k}: {type(v).__name__}")
        raise RuntimeError("Manual inspection needed.")
else:
    state_dict = raw
    print("Top-level is not a dict; using as-is.")

print(f"\nNumber of weight tensors: {len(state_dict)}")
print(f"\nFirst 20 keys (to understand naming convention):")
for k in list(state_dict.keys())[:20]:
    print(f"  {k}  {tuple(state_dict[k].shape)}")
print(f"\nLast 10 keys:")
for k in list(state_dict.keys())[-10:]:
    print(f"  {k}  {tuple(state_dict[k].shape)}")

## Cell 3 — Build ViT-Large/16 via timm + load weights

RETFound = ViT-Large/16 (patch_size=16, embed_dim=1024, depth=24, heads=16) pretrained with MAE on 1.6M retinal images.

We build it via timm with `num_classes=0` (no classification head — pure feature extractor). MAE pretrained models don't have a classification head anyway.

In [ ]:
import timm
print("timm version:", timm.__version__)

# Build ViT-L/16 with no classification head — pure feature extractor
model = timm.create_model(
    "vit_large_patch16_224",
    pretrained=False,
    num_classes=0,        # no head — outputs features only
    global_pool="",       # no global pooling — returns full token sequence
)

n_params = sum(p.numel() for p in model.parameters())
print(f"\nViT-L/16 architecture built: {n_params/1e6:.1f}M params")

# Try to load — strict=False so we see what doesn't match
missing, unexpected = model.load_state_dict(state_dict, strict=False)
print(f"\nLoad result:")
print(f"  Missing keys ({len(missing)}): {missing[:5]}{'...' if len(missing) > 5 else ''}")
print(f"  Unexpected keys ({len(unexpected)}): {unexpected[:10]}{'...' if len(unexpected) > 10 else ''}")

# Diagnose: if there are many missing/unexpected, the keys probably have a prefix mismatch
if len(missing) > 50:
    print("\n!! Large mismatch — checking if state_dict has a prefix to strip.")
    sd_keys = list(state_dict.keys())
    print(f"  State dict starts with: {sd_keys[0]}")
    print(f"  Model expects starts with: {list(model.state_dict().keys())[0]}")
    
    # Common fix: strip "module." or "encoder." prefix
    for prefix in ["module.", "encoder.", "model."]:
        if all(k.startswith(prefix) for k in sd_keys[:5]):
            print(f"\n  -> Detected prefix '{prefix}'. Stripping and retrying...")
            stripped = {k[len(prefix):]: v for k, v in state_dict.items() if k.startswith(prefix)}
            missing, unexpected = model.load_state_dict(stripped, strict=False)
            print(f"  After strip: missing={len(missing)}, unexpected={len(unexpected)}")
            break

model = model.to(DEVICE).eval()
print(f"\nModel on device: {next(model.parameters()).device}")

## Cell 4 — Load 5 BRSET sample images + preprocess

In [ ]:
# RETFound uses ImageNet normalization (standard for ViT pretrained with MAE)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMG_SIZE = 224

preprocess = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Grab 5 sample BRSET images
sample_paths = sorted(glob.glob(os.path.join(IMAGES_DIR, "*.jpg")))[:5]
print("Sample images:")
for p in sample_paths: print(" ", p)

batch = torch.stack([preprocess(Image.open(p).convert("RGB")) for p in sample_paths])
print(f"\nBatch shape: {batch.shape}  dtype={batch.dtype}  range=[{batch.min():.2f},{batch.max():.2f}]")

## Cell 5 — Forward pass + verify output shapes

**Two output modes we care about:**

- **Token-level features:** `forward_features(x)` returns `(B, 197, 1024)` — 196 patch tokens + 1 CLS token, each 1024-dim. This is what the patch-aggregator architecture will consume.
- **Pooled features:** `forward(x)` returns `(B, 1024)` after global-pool + (since num_classes=0) no head. This is the Chen et al. 2025 baseline configuration.

In [ ]:
x = batch.to(DEVICE)

with torch.no_grad():
    # Token-level features (full sequence)
    tokens = model.forward_features(x)
    # Pooled features (the Chen et al. setup)
    pooled = model(x)

print(f"forward_features output:")
print(f"  shape: {tokens.shape}  (expected: [5, 197, 1024] — 1 CLS + 196 patches × 1024 dim)")
print(f"  dtype: {tokens.dtype}")
print(f"  range: [{tokens.min():.3f}, {tokens.max():.3f}]")
print(f"  contains NaN: {torch.isnan(tokens).any().item()}")

print(f"\nforward output (pooled):")
print(f"  shape: {pooled.shape}  (expected: [5, 1024])")
print(f"  range: [{pooled.min():.3f}, {pooled.max():.3f}]")

# Sanity: features for different images should differ
sim = torch.cosine_similarity(pooled[0:1], pooled, dim=1)
print(f"\nCosine similarity of image 0 to images 0-4 (first should be 1.0, others < 1.0):")
print(f"  {sim.cpu().numpy().round(4)}")
assert sim[0].item() > 0.999, "image 0 vs itself should be ~1.0"
assert sim[1:].max().item() < 0.999, "different images should not be identical"
print("\n✅ RETFound loaded and producing reasonable features.")

## Cell 6 — Save a tiny verification artifact

Just so we have proof of life — a JSON with the verified output shapes and a few feature norms.

In [ ]:
out_dir = pathlib.Path("/kaggle/working/results")
out_dir.mkdir(exist_ok=True)

verification = {
    "weights_path": WEIGHTS,
    "model": "vit_large_patch16_224 (timm)",
    "n_params_M": round(n_params / 1e6, 2),
    "load_status": {
        "missing_keys": len(missing),
        "unexpected_keys": len(unexpected),
    },
    "input_shape": list(batch.shape),
    "forward_features_shape": list(tokens.shape),
    "forward_shape": list(pooled.shape),
    "feature_norms_per_image": pooled.norm(dim=1).cpu().tolist(),
    "sample_images_used": sample_paths,
    "ready_for_patch_aggregator": True,
}
with open(out_dir / "retfound_smoke_test.json", "w") as f:
    json.dump(verification, f, indent=2)
print(json.dumps(verification, indent=2))